## Experimental Setup

In this part of the project, we experimented with different transfer learning configurations using a pretrained ResNet50 model. The goal was to understand how fine-tuning, learning rate, and regularization affect model performance.

We used a pretrained ResNet50 backbone with ImageNet weights as the base for all experiments. Instead of training from scratch, this approach allows the model to leverage general visual features learned from a large dataset, and adapt them to our specific task of classifying paintings by artist.

Across five experiments, we varied three key factors:
- **Fine-tuning depth** (5 vs 10 top layers unfrozen)
- **Learning rate** (1e-3 vs 1e-5)
- **Dropout rate** (0.4, 0.5, 0.6)

All models used the same dataset splits, input resolution (224×224), and augmentation pipeline. Each was trained for up to 20 epochs with early stopping (patience=3) to prevent overfitting.

| Model | Fine-tune | Unfrozen Layers | LR   | Dropout |
|-------|-----------|-----------------|------|---------|
| 1     | Yes       | 5               | 1e-3 | 0.5     |
| 2     | Yes       | 10              | 1e-3 | 0.5     |
| 3     | Yes       | 10              | 1e-5 | 0.6     |
| 4     | Yes       | 5               | 1e-5 | 0.4     |
| 5     | Yes       | 10              | 1e-5 | 0.5     |

In [2]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger,
)

sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import build_datasets
from src.models.transfer import build_transfer_model

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

SPLITS_DIR = "../data/splits"
PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../results/models"
LOGS_DIR = "../results/logs"
FIGURES_DIR = "../results/figures"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [4]:
train_ds, val_ds, test_ds, NUM_CLASSES = build_datasets(
    splits_dir=SPLITS_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    augment_train=True,
    use_processed=True,
    processed_dir=PROCESSED_DIR
)

# One-hot encode labels (consistent with rest of project)
train_ds_oh = train_ds.map(lambda x, y: (x, tf.one_hot(y, NUM_CLASSES)))
val_ds_oh = val_ds.map(lambda x, y: (x, tf.one_hot(y, NUM_CLASSES)))
test_ds_oh = test_ds.map(lambda x, y: (x, tf.one_hot(y, NUM_CLASSES)))

print(f"Dataset loaded with {NUM_CLASSES} classes.")

Dataset loaded with 23 classes.


In [5]:
def get_callbacks(model_name):
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            mode="min",
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            mode="min",
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=f"{MODELS_DIR}/{model_name}.keras",
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        CSVLogger(f"{LOGS_DIR}/{model_name}.csv"),
    ]


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history["accuracy"], label="Train Accuracy")
    axes[0].plot(history.history["val_accuracy"], label="Val Accuracy")
    axes[0].set_title(f"{title} \u2014 Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history["loss"], label="Train Loss")
    axes[1].plot(history.history["val_loss"], label="Val Loss")
    axes[1].set_title(f"{title} \u2014 Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = f"{FIGURES_DIR}/{title.lower().replace(' ', '_')}_learning_curves.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {save_path}")
    plt.show()


def plot_history_comparison(histories, labels, save_name="transfer_comparison"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for history, label in zip(histories, labels):
        axes[0].plot(history.history["accuracy"], linestyle="--", label=f"{label} Train")
        axes[0].plot(history.history["val_accuracy"], label=f"{label} Val")
    axes[0].set_title("Accuracy Comparison")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    for history, label in zip(histories, labels):
        axes[1].plot(history.history["loss"], linestyle="--", label=f"{label} Train")
        axes[1].plot(history.history["val_loss"], label=f"{label} Val")
    axes[1].set_title("Loss Comparison")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/{save_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

## Models 1, 2

The goal of experimenting with Model 1 and Model 2 was to understand how the level of fine-tuning affects model performance.

Model 1 was designed with limited fine-tuning (only the top 5 layers unfrozen). This serves as a baseline to evaluate how well the pretrained features perform with minimal adaptation to the target task.

Model 2 increased the fine-tuning depth to 10 layers. The idea was to allow the model to adjust more of its internal representations to better capture the unique visual characteristics of each artist’s style.

In [6]:
model_1 = build_transfer_model(
    num_classes=NUM_CLASSES,
    fine_tune=True,
    unfreeze_top_layers=5,
)

history_1 = model_1.fit(
    train_ds_oh,
    validation_data=val_ds_oh,
    epochs=EPOCHS,
    callbacks=get_callbacks("transfer_model_1")
)


Epoch 1/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - accuracy: 0.3055 - f1_macro: 0.2484 - loss: 2.5020
Epoch 1: val_loss improved from None to 1.31280, saving model to ../results/models/transfer_model_1.keras

Epoch 1: finished saving model to ../results/models/transfer_model_1.keras
292/292 ━━━━━━━━━━━━━━━━━━━━ 124s 416ms/step - accuracy: 0.4116 - f1_macro: 0.3582 - loss: 2.0420 - val_accuracy: 0.6212 - val_f1_macro: 0.5572 - val_loss: 1.3128 - learning_rate: 0.0010
Epoch 2/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.6003 - f1_macro: 0.5591 - loss: 1.3339
Epoch 2: val_loss improved from 1.31280 to 1.10126, saving model to ../results/models/transfer_model_1.keras

Epoch 2: finished saving model to ../results/models/transfer_model_1.keras
292/292 ━━━━━━━━━━━━━━━━━━━━ 130s 444ms/step - accuracy: 0.6183 - f1_macro: 0.5829 - loss: 1.2797 - val_accuracy: 0.6762 - val_f1_macro: 0.6338 - val_loss: 1.1013 - learning_rate: 0.0010
Epoch 3/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 3

In [ ]:
model_2 = build_transfer_model(
    num_classes=NUM_CLASSES,
    fine_tune=True,
    unfreeze_top_layers=10,
)

history_2 = model_2.fit(
    train_ds_oh,
    validation_data=val_ds_oh,
    epochs=EPOCHS,
    callbacks=get_callbacks("transfer_model_2")
)

Epoch 1/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - accuracy: 0.3461 - f1_macro: 0.2942 - loss: 2.2902
Epoch 1: val_loss improved from None to 1.06219, saving model to ../results/models/transfer_model_2.keras

Epoch 1: finished saving model to ../results/models/transfer_model_2.keras
292/292 ━━━━━━━━━━━━━━━━━━━━ 106s 353ms/step - accuracy: 0.4805 - f1_macro: 0.4389 - loss: 1.7788 - val_accuracy: 0.6787 - val_f1_macro: 0.6317 - val_loss: 1.0622 - learning_rate: 0.0010
Epoch 2/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step - accuracy: 0.6563 - f1_macro: 0.6291 - loss: 1.1657
Epoch 2: val_loss improved from 1.06219 to 0.91195, saving model to ../results/models/transfer_model_2.keras

Epoch 2: finished saving model to ../results/models/transfer_model_2.keras
292/292 ━━━━━━━━━━━━━━━━━━━━ 101s 346ms/step - accuracy: 0.6710 - f1_macro: 0.6444 - loss: 1.1213 - val_accuracy: 0.7266 - val_f1_macro: 0.6998 - val_loss: 0.9120 - learning_rate: 0.0010
Epoch 3/20
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 28

In [ ]:
plot_history_comparison(
    [history_1, history_2],
    ["Model 1", "Model 2"],
    save_name="transfer_comparison_1_2"
)

Model 1

Model 1 achieved a high training accuracy, but its validation accuracy stayed much lower. The gap between training and validation performance shows that the model is overfitting. It learned the training data very well, but it did not generalize as effectively to unseen validation data. The training loss decreased steadily, which confirms the model kept fitting the training set closely.

Model 2

Model 2 showed stronger training accuracy, indicating that unfreezing more layers allowed the model to learn more complex patterns. However, the validation accuracy did not improve proportionally, and even decreased slightly toward the end, suggesting that some of the deeper layers may have overfitted to training-specific details.

Comparison: Both models show signs of overfitting, especially with LR = 1e-3. However, Model 1 (fewer unfrozen layers) had slightly better stability on the validation set. This suggests that a more conservative fine-tuning strategy might help, especially when combined with a lower learning rate.

Next step → reduce the learning rate to 1e-5 and experiment with dropout.

## Motivation for Models 3–5

After observing that Models 1 and 2 showed signs of overfitting with a higher learning rate, we focused on improving generalization rather than increasing model complexity.

For Models 3–5, we reduced the learning rate to 1e-5 to make training more stable and better preserve pretrained features. We also varied the dropout rate and fine-tuning depth to find the right balance between underfitting and overfitting.

| Model | Fine-tune | Unfrozen Layers | LR   | Dropout | Goal |
|-------|-----------|-----------------|------|---------|------|
| 3     | Yes       | 10              | 1e-5 | 0.6     | Stronger regularization |
| 4     | Yes       | 5               | 1e-5 | 0.4     | Less regularization, fewer layers |
| 5     | Yes       | 10              | 1e-5 | 0.5     | Middle ground |

## Model 3

In [ ]:
model_3 = build_transfer_model(
    num_classes=NUM_CLASSES,
    fine_tune=True,
    unfreeze_top_layers=10,
    dropout_rate=0.6,
    dense_units=256,
    learning_rate=1e-5,
)

history_3 = model_3.fit(
    train_ds_oh,
    validation_data=val_ds_oh,
    epochs=EPOCHS,
    callbacks=get_callbacks("transfer_model_3")
)

In [ ]:
plot_history(history_3, "Model 3")

Model 3

Model 3 gave the most balanced results overall. The training and validation accuracy curves stay very close to each other during training, which suggests that the model generalized well and did not overfit strongly. The loss curves are also smooth and consistent, showing stable learning behavior.

Compared to Models 1 and 2, Model 3 shows much less overfitting, which is likely due to the lower learning rate (1e-5) and higher dropout (0.6). This configuration preserved the pretrained features better while still allowing the model to adapt to the classification task.

## Model 4

Since the model with fewer trainable layers showed better validation performance, the next experiment keeps the fine-tuning depth small and instead adjusts the dropout rate. This helps test whether stronger regularization can improve generalization without increasing model complexity.

In [ ]:
model_4 = build_transfer_model(
    num_classes=NUM_CLASSES,
    fine_tune=True,
    unfreeze_top_layers=5,
    dropout_rate=0.4,
    dense_units=256,
    learning_rate=1e-5,
)

history_4 = model_4.fit(
    train_ds_oh,
    validation_data=val_ds_oh,
    epochs=EPOCHS,
    callbacks=get_callbacks("transfer_model_4")
)

In [ ]:
plot_history(history_4, "Model 4")

Model 4

Model 4 shows stable training behavior, and the validation accuracy is slightly higher than the training accuracy during most epochs. This is a good sign that the model is not overfitting. However, the final accuracy is lower than in Models 3 and 5, which means the model is likely underfitting slightly. With only 5 layers unfrozen and a low learning rate, the model may not have had enough capacity to fully adapt to the task.

## Model 5

In [ ]:
model_5 = build_transfer_model(
    num_classes=NUM_CLASSES,
    fine_tune=True,
    unfreeze_top_layers=10,
    dropout_rate=0.5,
    dense_units=256,
    learning_rate=1e-5,
)

history_5 = model_5.fit(
    train_ds_oh,
    validation_data=val_ds_oh,
    epochs=EPOCHS,
    callbacks=get_callbacks("transfer_model_5")
)

In [ ]:
plot_history(history_5, "Model 5")

Model 5

Model 5 achieved the highest training accuracy among Models 3, 4, and 5, and it also reached strong validation accuracy. This shows that the model has good learning capacity. However, the gap between training and validation performance started to increase in later epochs, and the validation loss began to rise slightly, which are early signs of overfitting. Despite this, it still performed better than Model 4 (underfitting) and was competitive with Model 3 (best generalization).

## Models 3–5 Comparison

In [ ]:
# Compare Models 3-5
plot_history_comparison(
    [history_3, history_4, history_5],
    ["Model 3", "Model 4", "Model 5"],
    save_name="transfer_comparison_3_4_5"
)

## Best Model — Test Set Evaluation

Select the best model (by validation loss) and evaluate on the held-out test set.

In [ ]:
import pandas as pd

# Identify best model by final val_loss
candidates = {
    "Model 3": (model_3, history_3),
    "Model 4": (model_4, history_4),
    "Model 5": (model_5, history_5),
}

best_name = min(candidates, key=lambda k: min(candidates[k][1].history["val_loss"]))
best_model = candidates[best_name][0]
print(f"Best transfer model: {best_name}")
print(f"  Best val_loss: {min(candidates[best_name][1].history['val_loss']):.4f}")

# Evaluate on test set
test_results = best_model.evaluate(test_ds_oh, verbose=1)
print(f"\n--- Test Set Results ({best_name}) ---")
for name, value in zip(best_model.metrics_names, test_results):
    print(f"  {name:15s}: {value:.4f}")

# Save the best model as the canonical transfer checkpoint
best_model.save(f"{MODELS_DIR}/transfer.keras")
print(f"\nBest model saved to {MODELS_DIR}/transfer.keras")